In [ ]:
# ==============================================================================
# 1. INSTALAÇÃO E IMPORTAÇÃO DE BIBLIOTECAS
# ==============================================================================
!pip install --quiet gspread pandas

import pandas as pd
import gspread
from google.colab import auth
from google.auth import default

# ==============================================================================
# 2. AUTENTICAÇÃO NO GOOGLE
# ==============================================================================
print("Autenticando sua conta do Google...")
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
print("Autenticação realizada com sucesso!\n")

# ==============================================================================
# 3. CONFIGURAÇÃO DA PLANILHA (ALTERE AQUI A URL E O NOME DA ABA)
# ==============================================================================
# Cole aqui a URL completa da sua planilha do Google Sheets:
URL_DA_PLANILHA = "https://docs.google.com/spreadsheets/d/1UW_gc_40EjyNpOZ5lnJoaSSXKmaOeZ-hnlrUmHThEo4/edit?gid=1426200946#gid=1426200946"

# Nome da aba que contém os dados originais:
NOME_DA_ABA_ORIGINAL = "analise_erros_categorizada"

# ==============================================================================
# 4. FUNÇÕES DE CÁLCULO E PERSISTÊNCIA
# ==============================================================================

def carregar_dados(url_planilha, nome_aba):
    """Abre a planilha e converte os dados em um DataFrame Pandas."""
    spreadsheet = gc.open_by_url(url_planilha)
    worksheet = spreadsheet.worksheet(nome_aba)
    dados = worksheet.get_all_records()
    return spreadsheet, pd.DataFrame(dados)

def calcular_percentual_por_questao(df, coluna_alvo):
    """Calcula a quantidade e o percentual agrupados por questão."""
    df_clean = df.dropna(subset=['questao', coluna_alvo]).copy()
    df_clean = df_clean[df_clean[coluna_alvo].astype(str).str.strip() != ""]

    agrupado = (
        df_clean.groupby(['questao', coluna_alvo])
        .size()
        .reset_index(name='quantidade')
    )

    total_por_questao = df_clean.groupby('questao').size().reset_index(name='total_questao')
    resultado = pd.merge(agrupado, total_por_questao, on='questao')
    resultado['percentual (%)'] = (resultado['quantidade'] / resultado['total_questao'] * 100).round(2)

    return resultado

def calcular_indice_geral(df, coluna_alvo):
    """Calcula a quantidade e o percentual geral para uma coluna simples."""
    df_clean = df.dropna(subset=[coluna_alvo]).copy()
    df_clean = df_clean[df_clean[coluna_alvo].astype(str).str.strip() != ""]

    total_geral = len(df_clean)

    geral = df_clean[coluna_alvo].value_counts().reset_index()
    geral.columns = [coluna_alvo, 'quantidade']
    geral['percentual_geral (%)'] = (geral['quantidade'] / total_geral * 100).round(2)

    return geral

def calcular_indice_geral_combinado(df, col1, col2):
    """Calcula a quantidade e o percentual geral para a combinação de duas colunas."""
    df_clean = df.dropna(subset=[col1, col2]).copy()
    df_clean = df_clean[
        (df_clean[col1].astype(str).str.strip() != "") &
        (df_clean[col2].astype(str).str.strip() != "")
    ]

    total_geral = len(df_clean)

    combinado = (
        df_clean.groupby([col1, col2])
        .size()
        .reset_index(name='quantidade')
        .sort_values(by='quantidade', ascending=False)
    )

    combinado['percentual_geral (%)'] = (combinado['quantidade'] / total_geral * 100).round(2)

    return combinado

def salvar_em_aba(spreadsheet, df_dados, nome_aba):
    """Cria uma nova aba (ou limpa se já existir) e salva o DataFrame nela."""
    try:
        worksheet = spreadsheet.worksheet(nome_aba)
        worksheet.clear()
        print(f"➜ Aba '{nome_aba}' atualizada com sucesso.")
    except gspread.exceptions.WorksheetNotFound:
        worksheet = spreadsheet.add_worksheet(title=nome_aba, rows="100", cols="20")
        print(f"➜ Aba '{nome_aba}' criada e preenchida com sucesso.")

    dados_para_inserir = [df_dados.columns.values.tolist()] + df_dados.astype(str).values.tolist()
    worksheet.update("A1", dados_para_inserir)

# ==============================================================================
# 5. EXECUÇÃO PRINCIPAL
# ==============================================================================

if __name__ == "__main__":
    # A. Carregar dados
    spreadsheet, df = carregar_dados(URL_DA_PLANILHA, NOME_DA_ABA_ORIGINAL)
    print(f"Total de registros carregados: {len(df)}\n")

    # B. Processar Cálculos POR QUESTÃO
    df_origem_questao = calcular_percentual_por_questao(df, 'origem_erro')
    df_causa_questao = calcular_percentual_por_questao(df, 'categoria_causa')
    df_analise_questao = calcular_percentual_por_questao(df, 'analise_critica')

    # C. Processar Cálculos ÍNDICE GERAL (Simples)
    df_geral_origem = calcular_indice_geral(df, 'origem_erro')
    df_geral_causa = calcular_indice_geral(df, 'categoria_causa')
    df_geral_analise = calcular_indice_geral(df, 'analise_critica')

    # D. Processar Cálculo ÍNDICE GERAL COMBINADO (Origem x Causa)
    df_geral_origem_causa = calcular_indice_geral_combinado(df, 'origem_erro', 'categoria_causa')

    # E. Exibir resultados no Colab
    print("=== 1. PERCENTUAL DE 'origem_erro' POR QUESTÃO ===")
    display(df_origem_questao)

    print("\n=== 2. PERCENTUAL DE 'categoria_causa' POR QUESTÃO ===")
    display(df_causa_questao)

    print("\n=== 3. PERCENTUAL DE 'analise_critica' POR QUESTÃO ===")
    display(df_analise_questao)

    print("\n=== 4. ÍNDICE GERAL: 'origem_erro' ===")
    display(df_geral_origem)

    print("\n=== 5. ÍNDICE GERAL: 'categoria_causa' ===")
    display(df_geral_causa)

    print("\n=== 6. ÍNDICE GERAL: 'analise_critica' ===")
    display(df_geral_analise)

    print("\n=== 7. ÍNDICE GERAL COMBINADO: 'origem_erro' x 'categoria_causa' ===")
    display(df_geral_origem_causa)

    # F. Salvar de volta nas abas da planilha
    print("\n" + "="*50)
    print("Salvando abas de relatórios na sua planilha do Google...")
    print("="*50)

    salvar_em_aba(spreadsheet, df_origem_questao, "Resumo_Origem_Questao")
    salvar_em_aba(spreadsheet, df_causa_questao, "Resumo_Causa_Questao")
    salvar_em_aba(spreadsheet, df_analise_questao, "Resumo_Analise_Questao")

    salvar_em_aba(spreadsheet, df_geral_origem, "Geral_Origem")
    salvar_em_aba(spreadsheet, df_geral_causa, "Geral_Causa")
    salvar_em_aba(spreadsheet, df_geral_analise, "Geral_Analise")

    salvar_em_aba(spreadsheet, df_geral_origem_causa, "Geral_Origem_x_Causa")

    print("\n✨ Concluído! Todas as 7 abas de relatório foram salvas na sua planilha do Google.")

Autenticando sua conta do Google...
Autenticação realizada com sucesso!

Total de registros carregados: 4632

=== 1. PERCENTUAL DE 'origem_erro' POR QUESTÃO ===


,questao,origem_erro,quantidade,total_questao,percentual (%)
0,T240Q01,AMBIGUIDADE,52,583,8.92
1,T240Q01,ERRO_DATASET,405,583,69.47
2,T240Q01,ERRO_MODELO,126,583,21.61
3,T240Q02,AMBIGUIDADE,59,531,11.11
4,T240Q02,ERRO_DATASET,281,531,52.92
5,T240Q02,ERRO_MODELO,191,531,35.97
6,T300Q01,AMBIGUIDADE,17,102,16.67
7,T300Q01,ERRO_DATASET,61,102,59.80
8,T300Q01,ERRO_MODELO,24,102,23.53
9,T300Q02,AMBIGUIDADE,53,558,9.50



=== 2. PERCENTUAL DE 'categoria_causa' POR QUESTÃO ===


,questao,categoria_causa,quantidade,total_questao,percentual (%)
0,T240Q01,Alucinação de Incomum / Superestimação de Orig...,77,583,13.21
1,T240Q01,Critério Humano Inconsistente / Subjetividade ...,126,583,21.61
2,T240Q01,Erro Óbvio de Gabarito,4,583,0.69
3,T240Q01,Falta de Contexto / Elipse,5,583,0.86
4,T240Q01,Rigidez Semântica / Literalidade,371,583,63.64
5,T240Q02,Alucinação de Incomum / Superestimação de Orig...,62,531,11.68
6,T240Q02,Critério Humano Inconsistente / Subjetividade ...,86,531,16.20
7,T240Q02,Erro Óbvio de Gabarito,16,531,3.01
8,T240Q02,Falta de Contexto / Elipse,17,531,3.20
9,T240Q02,Rigidez Semântica / Literalidade,350,531,65.91



=== 3. PERCENTUAL DE 'analise_critica' POR QUESTÃO ===


,questao,analise_critica,quantidade,total_questao,percentual (%)
0,T240Q01,A abordagem da poluição como crítica indireta ...,1,583,0.17
1,T240Q01,A ambiguidade surge da interpretação duvidosa ...,1,583,0.17
2,T240Q01,A analogia 'humanos como piolhos' é comum e nã...,1,583,0.17
3,T240Q01,A associação da Terra a uma emoção humana (fúr...,1,583,0.17
4,T240Q01,A avaliação depende de critérios subjetivos so...,1,583,0.17
...,...,...,...,...,...
4627,T570Q03,"O texto, embora simples, apresenta uma narrati...",1,598,0.17
4628,T570Q03,"O texto, embora simples, apresenta uma narrati...",1,598,0.17
4629,T570Q03,"O texto, embora simples, apresenta uma narrati...",1,598,0.17
4630,T570Q03,O uso de 'curto-circuito' como clichê é comum ...,1,598,0.17


... (18 outputs left out)

➜ Aba 'Geral_Analise' atualizada com sucesso.


/tmp/ipykernel_722/1229306418.py:102: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  worksheet.update("A1", dados_para_inserir)


➜ Aba 'Geral_Origem_x_Causa' criada e preenchida com sucesso.

✨ Concluído! Todas as 7 abas de relatório foram salvas na sua planilha do Google.


/tmp/ipykernel_722/1229306418.py:102: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  worksheet.update("A1", dados_para_inserir)
